# Tests — `p4tools.stats` uncertainty API

In [1]:
import numpy as np, pandas as pd
from p4tools.stats import add_uncertainty_columns, tile_quality, classify_tile_quality

## Layer 1 — deterministic, synthetic (no network)

In [2]:
# fan: pos_std = hypot(3,4)=5 ; size_cv = mean(20/100, 5/50)=0.15 ; angle always usable
fan = pd.DataFrame({'x_std':[3.0],'y_std':[4.0],'angle_std':[10.0],
                    'distance':[100.0],'distance_std':[20.0],
                    'spread':[50.0],'spread_std':[5.0],'n_votes':[7]})
r = add_uncertainty_columns(fan, min_votes=5)
assert r.pos_std.iloc[0] == 5.0
assert abs(r.size_cv.iloc[0] - 0.15) < 1e-12
assert bool(r.angle_usable.iloc[0]) is True
assert bool(r.scatter_ok.iloc[0]) is True
# returns a copy, does not mutate input
assert 'pos_std' not in fan.columns

In [3]:
# blotch: ratio radius_2/radius_1 = 0.9 (near-circular -> angle NOT usable) vs 0.5 (usable)
bl = pd.DataFrame({'x_std':[0.0,0.0],'y_std':[0.0,0.0],'angle_std':[1.0,1.0],
                   'radius_1':[10.0,10.0],'radius_2':[9.0,5.0],
                   'radius1_std':[1.0,1.0],'radius2_std':[1.0,1.0],'n_votes':[3,10]})
rb = add_uncertainty_columns(bl, min_votes=5, circular_ratio_limit=0.8)
assert bool(rb.angle_usable.iloc[0]) is False   # 0.9 > 0.8
assert bool(rb.angle_usable.iloc[1]) is True    # 0.5 <= 0.8
assert bool(rb.scatter_ok.iloc[0]) is False     # 3 < 5
assert bool(rb.scatter_ok.iloc[1]) is True       # 10 >= 5

## Layer 2/3 — integration invariants on v3.1

In [4]:
tq = tile_quality('fan', version='v3.1')
assert tq.index.name == 'tile_id'
assert (tq.n_scatter_markings >= 0).all()
# scatter is NaN exactly when no marking passed the vote gate
assert tq.loc[tq.n_scatter_markings == 0, 'pos_scatter'].isna().all()
assert tq.loc[tq.n_scatter_markings > 0, 'pos_scatter'].notna().all()
# ranks are within [0, 100]
for col in ['support_rank', 'scatter_rank']:
    s = tq[col].dropna()
    assert s.between(0, 100).all()

In [5]:
cls = classify_tile_quality(tq)
assert set(cls.quality_class.dropna().unique()) <= {'consistent','contested','sparse','noisy'}
# unclassifiable exactly where scatter_rank is missing (support_rank never is)
assert (cls.quality_class.isna() == cls.scatter_rank.isna()).all()

In [6]:
# classify requires rank columns
try:
    classify_tile_quality(pd.DataFrame({'foo':[1]}))
    assert False, 'should have raised'
except ValueError:
    pass
# bad kwargs rejected
for bad in [dict(kind='nope'), dict(agg='sum')]:
    try:
        tile_quality(version='v3.1', **bad); assert False
    except ValueError:
        pass